# Project 4: Customer Segmentation with K-Means
**Level:** Beginner to intermediate  
**Goal:** Group customers by annual income and spending score to support exploratory marketing analysis.

The dataset is synthetic and created in the notebook for repeatable practice. Clusters are exploratory patterns, not verified customer types.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEED = 42
rng = np.random.default_rng(SEED)
# Create five illustrative customer groups
centers = [(25,25),(25,75),(55,50),(85,20),(85,80)]
parts = []
for income, spend in centers:
    count = 80
    parts.append(pd.DataFrame({
        "annual_income_k": rng.normal(income, 8, count).clip(10, 120),
        "spending_score": rng.normal(spend, 9, count).clip(1, 100),
        "age": rng.integers(18, 70, count)
    }))
df = pd.concat(parts, ignore_index=True).round(1)
display(df.head())
display(df.describe().round(2))

## 1. Explore the customer data

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(df["annual_income_k"], df["spending_score"], alpha=0.65)
plt.xlabel("Annual income (thousand illustrative units)")
plt.ylabel("Spending score (1–100)")
plt.title("Income vs spending score")
plt.tight_layout(); plt.show()

## 2. Select a cluster count using silhouette score

In [ ]:
features = ["annual_income_k", "spending_score"]
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scores = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    scores.append({"k":k, "silhouette_score":silhouette_score(X_scaled, labels)})
score_df = pd.DataFrame(scores)
display(score_df.round(3))
plt.figure(figsize=(7,4))
plt.plot(score_df["k"], score_df["silhouette_score"], marker="o")
plt.xlabel("Number of clusters (k)"); plt.ylabel("Silhouette score")
plt.title("Choosing k"); plt.xticks(score_df["k"]); plt.tight_layout(); plt.show()

## 3. Fit final K-Means model

In [ ]:
# Select k based on the measured silhouette score. In practice, also use business context.
best_k = int(score_df.loc[score_df["silhouette_score"].idxmax(), "k"])
kmeans = KMeans(n_clusters=best_k, n_init=20, random_state=SEED)
df["cluster"] = kmeans.fit_predict(X_scaled)
print("Selected k:", best_k)
print("Silhouette score:", round(silhouette_score(X_scaled, df["cluster"]), 3))

## 4. Visualize clusters and summarize profiles

In [ ]:
plt.figure(figsize=(8,5))
for cluster_id in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cluster_id]
    plt.scatter(subset["annual_income_k"], subset["spending_score"],
                alpha=0.7, label=f"Cluster {cluster_id}")
plt.xlabel("Annual income (thousand illustrative units)")
plt.ylabel("Spending score (1–100)")
plt.title("Customer segments discovered by K-Means")
plt.legend(); plt.tight_layout(); plt.show()

profile = df.groupby("cluster").agg(
    customers=("cluster","size"),
    mean_income_k=("annual_income_k","mean"),
    mean_spending_score=("spending_score","mean"),
    mean_age=("age","mean")
).round(1)
profile

## 5. Assign a segment to a new customer

In [ ]:
new_customer = pd.DataFrame([{"annual_income_k": 60, "spending_score": 72}])
new_cluster = kmeans.predict(scaler.transform(new_customer[features]))[0]
print("Assigned cluster:", int(new_cluster))
print("Cluster profile:")
display(profile.loc[[new_cluster]])

## 6. Interpretation and limitations
- K-Means is sensitive to scaling, outliers, and the selected number of clusters.
- Silhouette score is one diagnostic; interpret clusters alongside domain knowledge and stability checks.
- These synthetic groups do not represent real customer behavior.
- For real customer data, minimize personal data, document consent and permitted use, and test whether segments are stable and useful.